In [417]:
from langchain_ollama import ChatOllama
import pandas as pd 
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate , ChatPromptTemplate
from langchain_core.tools import tool
from dotenv import load_dotenv
from pydantic import BaseModel , Field
import sqlite3
from datetime import date
from typing import Optional
from langchain.agents import create_agent
import matplotlib.pyplot as plt 
import seaborn as sns 
from connection import get_connection

In [418]:
import os
api_key =  os.getenv("NVIDIA_API_KEY")

In [419]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
client = ChatNVIDIA(
  model="meta/llama-3.3-70b-instruct",
  api_key=api_key, 
  temperature=0.2,
  top_p=0.7,
  max_completion_tokens = 1024,
)

In [420]:
# conn = sqlite3.connect("MoneyWise.db" , check_same_thread=False)
# cursor = conn.cursor()

current_date = date.today()

In [421]:
load_dotenv ()
# model = ChatOllama(model = 'qwen3.5:9b')

True

In [422]:
schema = """

Table: Transactions

Columns:
- Id INTEGER
- Title TEXT
- Amount REAL
- Category TEXT
- Type TEXT
- Mode TEXT
- Date TEXT

Table: Goals

Columns:
- Id INTEGER
- Title TEXT ,
- Started_At TEXT ,
- Deadline TEXT ,
- Target_Amount INTEGER ,
- Saved_Amount INTEGER ,
- Status TEXT

"""

In [423]:
class Financecommand(BaseModel):
    
    Operation: str = Field(
        description="""
Type of action user wants to perform.

Allowed values:
Add
Fetch
Update
Delete

Examples:
Bought books worth 120 -> Add
Fetch all education expenses -> Fetch
Update books purchase amount to 300 -> Update
Delete pizza expense -> Delete
"""
    )
    Title: Optional[str] = Field(
        default= None ,
        description="""
Short clear transaction title.

Examples:
apple shares purchase, books purchase, hotel dinner, salary received,
pocket money, petrol refill, freelancing payment
"""
    )

    Amount: Optional[int] = Field(
        default=None,
        description="""
Final transaction amount in numeric form only.

Rules:
- If quantity × price is given, calculate total amount.
Example:
4 shares each 51 rs = 204

Examples:
120, 500, 204, 25000
"""
    )

    Category: Optional[str] = Field(
        default=None,
        description="""
Smart category field.

Use only categories from the allowed list.
Never create random/custom categories.
If no clear match, use Other.

For Expense transactions:

Food
Groceries
Transport
Education
Shopping
Entertainment
Healthcare
Bills
Travel
Investment
Subscription
Other

Important Rules:
- bought shares, purchased stocks, mutual fund invested, SIP paid, crypto bought = Investment
- books, notebook, pen, stationery items = Education
- pizza, lunch, dinner, snacks, coffee = Food
- vegetables, fruits, rice, milk = Groceries
- petrol, fuel, uber, bus, train = Transport
- movie, bowling, gaming = Entertainment
- doctor, medicine, hospital = Healthcare
- electricity, internet, mobile recharge = Bills
- flight, hotel trip = Travel
- netflix, spotify, prime = Subscription


For Income transactions:

Salary
Pocket Money
Freelancing
Gift
Refund
Cashback
Investment
Other

Important Rules:
- father gave money / mother gave money = Pocket Money
- salary credited = Salary
- freelance payment = Freelancing
- returned money = Refund
- dividend / interest / profit = Investment
- gift money = Gift


Examples:
Bought Apple shares -> Investment
Received dividend -> Investment
Father gave me 500 -> Pocket Money
Got cashback of 50 -> Cashback


For Fetch / Update / Delete queries:

If user mentions category words, extract into Category.

Examples:
fetch education expenses -> Education
show food expenses -> Food
delete grocery expense -> Groceries
update transport expense amount -> Transport
fetch investments -> Investment
show salary entries -> Salary
"""
    )

    Mode: Optional[str] = Field(
        default= None ,
        description="""
Payment mode.

Examples:
Cash, Online, UPI, Credit Card, Debit Card, Wallet.

If not mentioned -> Online
"""
    )

    Type: Optional[str] = Field(
    default=None,
    description="""
Transaction nature.

Allowed values:
Expense
Income

Rules for Add operations:
Expense = money went out from user.
Income = money came to user.

Expense Examples:
bought books worth 120
paid rent 8000
purchased 4 Apple shares each 51 rs
invested 5000 in mutual fund
ordered pizza for 300

Income Examples:
received salary 25000
father gave me 500
friend returned 300
got dividend 120

Important Rule:
Buying shares/stocks/investments = Expense

Rules for Fetch / Update / Delete:
- If user clearly specifies expense or income, extract it.
- If not clearly mentioned, return None.

Examples:
fetch all expenses -> Expense
show income records -> Income
update pizza expense amount to 500 -> Expense
delete salary entry -> Income
fetch education records -> None
update books purchase amount to 300 -> None

Never guess when not specified for Fetch / Update / Delete.
"""
    )

    field: Optional[str] = Field(
        default = None ,
        description = """Name of the database column that needs to be updated.

Allowed values:
Title
Amount
Category
Mode
Type

Examples:
update amount of pizza purchase to 500 -> Amount
change category of books purchase to Education -> Category
update mode of rent payment to Cash -> Mode
rename pocket money title to Monthly Pocket Money -> Title
change type of salary entry to Income -> Type
For Delete / Update / Fetch queries, use user mentioned item name as Title when applicable.
"""


    )
    new_value : Optional[str] = Field(
        default=None,
        description="""New value that should replace the old existing value in the selected field.

Examples:
500
Education
Cash
Online
Income
Monthly Pocket Money

Rules:
- If field is Amount, return numeric value only.
- If field is Mode, use values like Cash, Online, UPI, Credit Card, Debit Card.
- If field is Type, use only Expense or Income.
- If field is Category, return proper category name.
- If field is Title, return short meaningful title.
"""
    )


In [424]:
class Goalcommand(BaseModel):
    Operation: Optional[str] = Field(
        default=None,
        description="""
Type of action user wants to perform on a goal.

Allowed values:
Create
Update
Delete
Fetch
"""
    )

    Title: Optional[str] = Field(
        default=None,
        description="""
Short clear name of the financial goal.

Create concise meaningful title from user message.

Examples:
buy shoes worth 2000 by aug -> Buy Shoes
save for iphone 15 -> Buy iPhone 15
trip to Goa next year -> Goa Trip
build emergency fund of 50000 -> Emergency Fund
buy gaming laptop -> Gaming Laptop
pay college fees -> College Fees
new bike purchase -> Buy Bike
"""
    )

    Started_at: Optional[date] = Field(
        default=None,
        description="""
Starting date of the goal.

Rules:
- If creating a goal and no start date is mentioned, use current date.
- If not applicable, return null.
- Return date only in YYYY-MM-DD format.

Examples:
today -> 2026-04-30
1 May 2026 -> 2026-05-01
next month -> first date of next month
"""
    )

    Deadline: Optional[date] = Field(
        default=None,
        description="""
Final target date / deadline by which the goal should be completed.

Rules:
- Return date only in YYYY-MM-DD format.
- Convert natural language dates into exact valid date.
- Use last reasonable date of that period.
- If not mentioned, return null.

Examples:
August 2026 -> 2026-08-31
31 Dec 2026 -> 2026-12-31
next March -> 2027-03-31
within 6 months -> calculated future date
"""
    )

    Target_Amount: Optional[int] = Field(
        default=None,
        description="""
Total money required to complete the goal.

Examples:
buy shoes worth 2000 -> 2000
save 50000 for emergency fund -> 50000
iphone goal of 80000 -> 80000
trip budget 15000 -> 15000

Extract numeric amount only.
"""
    )

    Saved_Amount: Optional[int] = Field(
        default=0,
        description="""
Amount already saved or newly added toward the goal.

Examples:
add 500 to shoes goal -> 500
saved 2000 for bike -> 2000
increase emergency fund by 1000 -> 1000

For create operations:
If not mentioned, default is 0.
"""
    )

    Status: Optional[str] = Field(
        default=None,
        description="""
Current status of the goal.

Allowed values:
Active
Completed
Failed
Paused

Rules:
- If Saved_Amount >= Target_Amount -> Completed
- Newly created goals usually Active
- User can explicitly mark completed or paused
"""
    )

    field: Optional[str] = Field(
        default=None,
        description="""
Name of the Goals table column that needs to be updated.

Allowed values:
Title
Started_at
Deadline
Target_Amount
Saved_Amount
Status

Examples:
change iphone goal target amount to 95000 -> Target_Amount
add 5000 to bike goal savings -> Saved_Amount
rename shoes goal to Running Shoes -> Title
change camera goal deadline to december end -> Deadline
pause emergency fund goal -> Status
change trip goal start date to next month -> Started_at

Rules:
- Use only one field name.
- For Update queries, identify what user wants to change.
"""
    )

    new_value: Optional[str] = Field(
        default=None,
        description="""
New value that should replace the old existing value in the selected field.

Examples:
95000
5000
Running Shoes
2026-12-31
Paused
2026-05-01

Rules:
- If field is Target_Amount or Saved_Amount, return numeric value only.
- If field is Deadline or Started_at, return date in YYYY-MM-DD format.
- If field is Status, use only Active, Completed, Paused, Failed.
- If field is Title, return short meaningful title.
"""
    )
    

In [425]:
parser = PydanticOutputParser(pydantic_object= Financecommand)
parser2 = PydanticOutputParser(pydantic_object=Goalcommand)

In [426]:
template1 = PromptTemplate(
    template="""
You are a smart finance transaction extractor.

Your job is to read the user's message and convert it into structured transaction data.

Extract these fields.
1. Operation
2. Title
3. Amount
4. Category
5. Mode
6. Type

Field Rules:

1. Operation:

Detect what user wants to do.

Allowed values:
Add
Fetch
Update
Delete

Examples:
Bought pizza for 300 -> Add
Father gave me 500 -> Add
Fetch all education expenses -> Fetch
Show cash expenses -> Fetch
Update books purchase amount to 500 -> Update
Delete pizza expense -> Delete


2. Title:

Create a short clear meaningful transaction title only if relevant.

Examples:
Bought pizza -> Pizza Purchase
Father gave pocket money -> Pocket Money
Received salary -> Salary Received
Paid electricity bill -> Electricity Bill

Extract the item/service name from the sentence.

Patterns:
"spent X on a trimmer"     → Trimmer Purchase
"spent X on medicines"     → Medicines  
"bought X"                 → X Purchase
"paid X for electricity"   → Electricity Bill
"got X as salary"          → Salary Received

NEVER ask user for the item name if it is already mentioned in the sentence.


3. Amount:

Extract only numeric amount when mentioned.

Examples:
₹500 -> 500
1,200 -> 1200


4. Category:

For Expense:
Use only the allowed categories:

Food
Groceries
Transport
Education
Shopping
Entertainment
Healthcare
Bills
Travel
Subscription
Investment
Other

Examples:
pizza -> Food
burger -> Food
vegetables -> Groceries
milk -> Groceries
petrol -> Transport
uber -> Transport
pen -> Education
books -> Education
movie tickets -> Entertainment
medicine -> Healthcare
electricity bill -> Bills
flight ticket -> Travel
netflix -> Subscription
shares -> Investment
mutual fund -> Investment


For Income:
Use only the allowed categories:

Salary
Pocket Money
Freelancing
Business
Gift
Refund
Cashback
Investment
Other

Examples:
salary credited -> Salary
father gave money -> Pocket Money
mother sent money -> Pocket Money
freelancing payment -> Freelancing
shop profit -> Business
gift money -> Gift
friend returned money -> Refund
cashback received -> Cashback
dividend received -> Investment
interest credited -> Investment
rent received -> Business

5. Mode:

Payment method.

Examples:
cash, online, UPI, credit card, debit card, wallet

Rules:
- If Operation = Add and mode not mentioned -> Online
- If Operation = Fetch / Update / Delete and mode not mentioned -> None


6. Type:

Expense = user spent money
Income = user received money

Examples:
Bought pizza for 300 -> Expense
Paid rent 8000 -> Expense
Father gave me 500 -> Income
Received salary 25000 -> Income

Rules:
- Buying shares/stocks/investment = Expense
- For Fetch queries fill Type only when clearly implied.

For Update operations:

Extract these additional fields:

1. field  
Name of the column user wants to modify.

Allowed values:
Date, Title, Amount, Category, Mode, Type

Examples:
update amount of pizza purchase to 500 -> field = Amount
change category of books purchase to Education -> field = Category
update payment mode of rent to Cash -> field = Mode
rename pocket money to Monthly Pocket Money -> field = Title
change salary entry type to Income -> field = Type


2. new_value  
The new value that should replace old value.

Examples:
500
Education
Cash
Online
Income
Monthly Pocket Money

Rules:
- If field = Amount, extract numeric only.
- If field = Type, use Expense or Income.
- If field = Mode, use Cash / Online / UPI / Debit Card / Credit Card.
- If field = Category, return proper category name.
- If field = Title, return short meaningful title.


Examples of full update queries:

update amount of books purchase to 300
-> Operation = Update
-> Title = Books Purchase
-> field = Amount
-> new_value = 300

change category of pizza purchase to Food
-> Operation = Update
-> Title = Pizza Purchase
-> field = Category
-> new_value = Food

rename pocket money to Monthly Pocket Money
-> Operation = Update
-> Title = Pocket Money
-> field = Title
-> new_value = Monthly Pocket Money

For Delete operations:

Extract the transaction title or identifying phrase the user wants removed.

Examples:

delete books purchase
-> Operation = Delete
-> Title = Books Purchase

delete pizza expense
-> Operation = Delete
-> Title = Pizza Purchase

remove salary received
-> Operation = Delete
-> Title = Salary Received

delete pocket money entry
-> Operation = Delete
-> Title = Pocket Money

delete Apple shares purchase
-> Operation = Delete
-> Title = Apple Shares Purchase

Rules:
- Words after delete/remove usually represent Title.
- Keep unrelated fields null unless explicitly mentioned.
- Create short meaningful title.


Important Rules:
- Understand natural language.
- Return only structured output.
- Do not explain anything.
- Do not return markdown.
- Always fill best possible values.
- For Fetch queries do not assume unnecessary fields.

CRITICAL RULE — Type field:
Type is ALWAYS only "Income" or "Expense". Nothing else.
Category is where the sub-type goes.

WRONG: Type=Cashback  RIGHT: Type=Income,  Category=Cashback
WRONG: Type=Salary    RIGHT: Type=Income,  Category=Salary
WRONG: Type=Refund    RIGHT: Type=Income,  Category=Refund

got cashback of 180        -> Type=Income, Category=Cashback
friend returned 2000       -> Type=Income, Category=Refund
my father gave me 500 cash -> Type=Income, Category=Pocket Money

User: spent 699 on a trimmer
Tool: add_transaction(Title="Trimmer Purchase", Amount=699, Type="Expense", Category="Shopping", Mode="Online")

User: spent 200 on a book
Tool: add_transaction(Title="Book Purchase", Amount=200, Type="Expense", Category="Education", Mode="Online")

User: spent 1500 on medicines
Tool: add_transaction(Title="Medicines", Amount=1500, Type="Expense", Category="Healthcare", Mode="Online")

User Input: {user_query}

{format_instructions}

Return only JSON.
""",
    input_variables=["user_query"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [427]:
template2 = PromptTemplate(
    template="""
You are a smart financial goal extractor.

Your job is to read the user's message and convert it into structured goal data.

Extract these fields:

1. Operation
2. Title
3. Started_at
4. Deadline
5. Target_Amount
6. Saved_Amount
7. Status


Field Rules:

1. Operation

Detect what user wants to do.

Allowed values:
Create
Update
Delete
Fetch

If user says:
fetch all marriage fund goals
show all bike goals
list phone goals

Then:
Operation = Fetch
Title = meaningful phrase before words goal/goals

For Update operations:
- Extract only fields user wants to change.
- Do not fill unrelated fields with defaults.
- Do not set Status unless user explicitly mentions it.
- Use Title to identify the goal.
- New values like Target_Amount or Saved_Amount are update values, not search conditions.

Examples:
marriage fund goals -> Marriage Fund
bike goals -> Bike
phone goal -> Phone

2. Title

Create short meaningful goal title.


3. Started_at



Goal start date.

Rules:
- If Operation = Create and no explicit start date is mentioned, always use today's current date.
- Even if deadline exists, Started_at should still be today's date unless user gives another start date.
- For Update/Delete/Fetch and no date mentioned -> null
- Return in YYYY-MM-DD format


4. Deadline

Goal completion deadline.

Relative month phrases must be converted to exact dates.

Examples:
next march -> 2027-03-31
this march -> 2026-03-31
next january -> next upcoming January end date
march 2027 -> 2027-03-31

Rules:
- Return date only in YYYY-MM-DD format
- Convert natural language dates into exact valid date
- Use last reasonable date of that period
- Use null if not mentioned

Examples:
by August 2026 -> 2026-08-31
before Diwali -> convert nearest valid date
within 6 months -> calculate future date
31 Dec 2026 -> 2026-12-31
next March -> 2027-03-31

Do not return text like:
by August
soon
next month
June end
Recognize short month names also.

Examples:
jan = january
feb = february
mar = march
apr = april
aug = august
sep = september
oct = october
nov = november
dec = december

Examples:
by jan next year -> 2027-01-31
by aug end -> 2026-08-31
by dec this year -> 2026-12-31


5. Target_Amount

Total required amount.


6. Saved_Amount

Money already saved or newly added.

If Create and not mentioned -> 0


7. Status

Current state of the goal.

Allowed values:
Active      -> Goal is in progress
Completed   -> Goal target achieved
Paused      -> Goal temporarily stopped
Failed      -> Deadline passed or goal abandoned

Rules:
- Newly created goals -> Active
- If Saved_Amount >= Target_Amount -> Completed
- If user says pause/hold/stop temporarily -> Paused
- If user says cancel/failed/missed deadline -> Failed
- If not specified, default -> Active

For Update operations:

Extract these additional fields:

1. field
Name of the Goals table column user wants to modify.

Allowed values:
Title
Started_At
Deadline
Target_Amount
Saved_Amount
Status

Examples:
change iphone goal target amount to 95000 -> field = Target_Amount
add 5000 to bike goal savings -> field = Saved_Amount
rename shoes goal to Running Shoes -> field = Title
change camera goal deadline to december end -> field = Deadline
pause emergency fund goal -> field = Status
change trip goal start date to next month -> field = Started_At


2. new_value
The new value that should replace old value.

Examples:
95000
5000
Running Shoes
2026-12-31
Paused
2026-05-01

Rules:
- If field = Target_Amount or Saved_Amount, extract numeric only.
- If field = Deadline or Started_At, return date in YYYY-MM-DD format.
- If field = Status, use only Active / Completed / Paused / Failed.
- If field = Title, return short meaningful title.


Examples of full update queries:

change iphone 15 goal target amount to 95000
-> Operation = Update
-> Title = Buy iPhone 15
-> field = Target_Amount
-> new_value = 95000

add 5000 to bike goal
-> Operation = Update
-> Title = Save for Bike
-> field = Saved_Amount
-> new_value = 5000

rename shoes goal to Running Shoes
-> Operation = Update
-> Title = Buy Shoes
-> field = Title
-> new_value = Running Shoes

pause emergency fund goal
-> Operation = Update
-> Title = Emergency Fund
-> field = Status
-> new_value = Paused


For Delete operations:

Extract the goal title or identifying phrase user wants removed.

Examples:

delete shoes goal
-> Operation = Delete
-> Title = Buy Shoes

remove iphone goal
-> Operation = Delete
-> Title = Buy iPhone 15

delete bike savings goal
-> Operation = Delete
-> Title = Save for Bike

delete goa trip goal
-> Operation = Delete
-> Title = Goa Trip

remove emergency fund
-> Operation = Delete
-> Title = Emergency Fund

For every Update query, field and new_value are mandatory.

Examples:
change iphone 15 goal target amount to 950000
-> field = Target_Amount
-> new_value = 950000

add 5000 to bike goal
-> field = Saved_Amount
-> new_value = 5000

pause emergency fund goal
-> field = Status
-> new_value = Paused


Rules:
- Words after delete/remove usually represent Title.
- Keep unrelated fields null unless explicitly mentioned.
- Create short meaningful title.


Important Rules:
- Understand natural language.
- Return only structured output.
- Do not explain anything.
- Do not return markdown.
- Always fill best possible values.
- For Fetch queries do not assume unnecessary fields.


Important Rules:

- Return only structured output.
- Do not explain anything.
- Do not return markdown.
- Use null where field is not applicable.
- Understand natural language.
- Always create best possible title.
- All dates must be in YYYY-MM-DD format only.
- If Operation = Update:
    Only changed fields should have values.
    All untouched fields must be null.

User Input: {user_query2}

{format_instructions}

Return only JSON.
""",
    input_variables=["user_query2"],
    partial_variables={
        "format_instructions": parser2.get_format_instructions()
    }
)

In [428]:
template3 = ChatPromptTemplate.from_template("""
You are a SQLite expert.

Database Schema:
{schema}

Rules:
- Only generate SQLite SQL
- Only use SELECT queries
- Do not use DELETE, DROP, UPDATE, ALTER
- Return ONLY SQL query
- No markdown

User Question:
{question}
""")


In [429]:
# transaction = "give me all the transactions which included groceries and was paid in cash"

In [430]:
# goal = "change iphone 15 goal target amount to 950000"

In [431]:
# chain = template1 | model | parser
# final_result = chain.invoke({'user_query' : transaction})

In [432]:
# chain_goal = template2 | model | parser2
# final_result_goal = chain_goal.invoke({'user_query2' : goal})

In [433]:
# print(final_result)

In [434]:
# print(final_result_goal)

## Tools

In [435]:
INCOME_CATS  = {"Salary", "Pocket Money", "Freelancing", "Business", "Gift", "Refund", "Cashback"}
EXPENSE_CATS = {"Food","Groceries","Transport","Education","Shopping",
                "Entertainment","Healthcare","Bills","Travel","Subscription",
                "Investment","Shares","Other"}
def _infer_type(Type, Category):
    if Type is not None:
        return Type
    if Category in INCOME_CATS:
        return "Income"
    if Category in EXPENSE_CATS:
        return "Expense"
    return None

In [436]:
@tool
def generate_and_execute_sql(question: str):
    """
    Generates and executes a read-only SQL query from a natural language question.

    Args:
        question (str): User's natural language query.

    Returns:
        dict: Contains generated SQL query and query results.
    """

    conn = None

    try:
        conn = get_connection()
        cursor = conn.cursor()

        chain = template3 | client

        response = chain.invoke({
            "schema": schema,
            "question": question
        })

        sql_query = response.content.strip()

        sql_query = (
            sql_query
            .replace("```sql", "")
            .replace("```", "")
            .strip()
        )

        upper_query = sql_query.upper()

        allowed_keywords = ["SELECT", "WITH"]

        if not upper_query.startswith(tuple(allowed_keywords)):
            return {
                "error": "Only read-only SELECT queries are allowed."
            }

        blocked_keywords = [
            "INSERT",
            "UPDATE",
            "DELETE",
            "DROP",
            "ALTER",
            "TRUNCATE",
            "CREATE",
            "REPLACE",
            "PRAGMA"
        ]

        if any(keyword in upper_query for keyword in blocked_keywords):
            return {
                "error": "Unsafe query detected."
            }

        cursor.execute(sql_query)

        rows = cursor.fetchall()

        columns = [desc[0] for desc in cursor.description]

        formatted_rows = [
            dict(zip(columns, row))
            for row in rows
        ]

        if not formatted_rows:
            return {
                "query": sql_query,
                "result": "No records found."
            }

        return {
            "query": sql_query,
            "result": formatted_rows
        }

    except Exception as e:
        return {
            "error": str(e)
        }

    finally:
        if conn:
            conn.close()

In [437]:
@tool

def add_transaction(Title=None, Amount=None, Type=None, Category=None, Mode=None):
    """
    Add a new financial transaction to the Transactions table.

    Use this function to store income or expense entries such as salary,
    groceries, rent, transport, shopping, etc.

    Args:
        Title (str): Short name or description of the transaction.
            Example: "Pizza", "Salary", "Electric Bill"

        Amount (float | int): Transaction amount.
            Must be a positive numeric value.

        Type (str): Type of transaction.
            Allowed values: "Income", "Expense"

        Category (str): Category of transaction.
            Example:
            Expense -> food, transport, bills, shopping
            Income -> salary, freelance, gift

        Mode (str): Payment mode used.
            Example: Cash, UPI, Card, Bank Transfer

    Returns:
        str: Success message after inserting the transaction.

    Behavior:
        - Automatically stores the current date.
        - Saves the transaction in the database.
        - Commits changes immediately.

    Example:
        add_transaction(
            Title="Burger",
            Amount=250,
            Type="Expense",
            Category="Food",
            Mode="UPI"
        )
    """
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        # FIX 1A — Guard: never insert a transaction without an amount
        if Amount is None:
            return "Error: Amount is required to add a transaction. Please provide the amount."

        resolved_type = _infer_type(Type, Category)

        if resolved_type is None:
            return "Error: Could not determine Type (Income/Expense). Please specify."

        today = date.today()

        # FIX 1B — was: 5 columns + 6 placeholders + no values tuple → runtime crash
        cursor.execute(
            """
            INSERT INTO Transactions
            (Date, Title, Amount, Type, Category, Mode)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (today, Title, Amount, resolved_type, Category, Mode or "Online")
        )

        conn.commit()

        return "Transaction added successfully."

    finally: 
        if conn:
            conn.close()

    # ? are placeholders which prevent SQL injection — called a parameterized query


In [438]:
def get_transactions_raw(
    Title=None, Amount=None, Type=None, Category=None, Mode=None,
    Date=None, Month=None, DateFrom=None, DateTo=None,
    AmountMin=None, AmountMax=None
):
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        query = "SELECT * FROM transactions WHERE 1=1"
        values = []

        # FIX 2A — exact date match (for update/delete targeting)
        if Date:
            query += " AND Date = ?"
            values.append(Date)

        # FIX 2B — month filter: e.g. Month='2026-01' matches all of January
        if Month:
            query += " AND strftime('%Y-%m', Date) = ?"
            values.append(Month)

        # FIX 2C — date range filter
        if DateFrom:
            query += " AND Date >= ?"
            values.append(DateFrom)
        if DateTo:
            query += " AND Date <= ?"
            values.append(DateTo)

        if Title:
            query += " AND LOWER(title) LIKE LOWER(?)"
            values.append(f"%{Title}%")

        # Exact amount (kept for precise targeting in update/delete)
        if Amount is not None:
            query += " AND amount = ?"
            values.append(Amount)

        # FIX 2D — amount range filters for fetch queries
        if AmountMin is not None:
            query += " AND amount >= ?"
            values.append(AmountMin)
        if AmountMax is not None:
            query += " AND amount <= ?"
            values.append(AmountMax)

        if Type:
            query += " AND lower(type) LIKE lower(?)"
            values.append(f"%{Type}%")

        if Category:
            query += " AND lower(category) LIKE lower(?)"
            values.append(f"%{Category}%")

        if Mode:
            query += " AND lower(mode) LIKE lower(?)"
            values.append(f"%{Mode}%")

        cursor.execute(query, values)
        return cursor.fetchall()
    finally:
        if conn:
            conn.close()


In [439]:
@tool

def get_transactions(
    Title: str = None,
    Amount: int = None,
    Type: str = None,
    Category: str = None,
    Mode: str = None,
    # FIX 3 — new date and amount-range params
    Date: str = None,
    Month: str = None,
    DateFrom: str = None,
    DateTo: str = None,
    AmountMin: int = None,
    AmountMax: int = None,
) -> list:
    """
    Retrieve transactions from the database using optional filters.

    Use this function to search and view transaction records such as
    income, expenses, category-based spending, or payment mode history.

    Args:
        Title (str, optional): Search by transaction title or name.
            Example: "Pizza", "Salary"

        Amount (int, optional): Search by exact transaction amount.

        Type (str, optional): Filter by transaction type.
            Allowed values: "Income", "Expense"

        Category (str, optional): Filter by category.
            Example: Food, Transport, Bills, Shopping, Salary

        Mode (str, optional): Filter by payment mode.
            Example: Cash, UPI, Card, Bank Transfer

        Date (str, optional): Filter by exact date in YYYY-MM-DD format.

        Month (str, optional): Filter by month in YYYY-MM format.
            Example: '2026-01' returns all January 2026 transactions.

        DateFrom (str, optional): Start of date range in YYYY-MM-DD format.

        DateTo (str, optional): End of date range in YYYY-MM-DD format.

        AmountMin (int, optional): Minimum amount (inclusive).

        AmountMax (int, optional): Maximum amount (inclusive).

    Returns:
        list: Matching transaction records from the database.

    Behavior:
        - If filters are provided, only matching records are returned.
        - If no filters are provided, all transactions are returned.
        - Useful for history tracking, summaries, and analytics.

    Example:
        get_transactions(Type="Expense", Category="Food")
        get_transactions(Month="2026-01", Type="Income")
        get_transactions(AmountMin=1000, Type="Expense")
        get_transactions(DateFrom="2026-02-01", DateTo="2026-03-31")
    """
    return get_transactions_raw(
        Title=Title, Amount=Amount, Type=Type, Category=Category, Mode=Mode,
        Date=Date, Month=Month, DateFrom=DateFrom, DateTo=DateTo,
        AmountMin=AmountMin, AmountMax=AmountMax
    )

# used LIKE for partial matching; strftime for month-based filtering


In [ ]:
@tool

def delete_transactions(Title: str = None, Amount: int = None, Type: str = None,
                        Category: str = None, Mode: str = None) -> str:
    """
    Delete transaction records from the database using optional filters.

    Use this function to remove incorrect, duplicate, or unwanted
    income and expense entries.

    Args:
        Title (str, optional): Match transaction title or name.
            Example: "Pizza", "Salary"

        Amount (int, optional): Match exact transaction amount.

        Type (str, optional): Filter by transaction type.
            Allowed values: "Income", "Expense"

        Category (str, optional): Filter by transaction category.
            Example: Food, Transport, Bills, Shopping

        Mode (str, optional): Filter by payment mode.
            Example: Cash, UPI, Card, Bank Transfer

    Returns:
        str: Status message after deletion.

    Behavior:
        - If no matching transaction is found, returns an error message.
        - If exactly one match is found, deletes it automatically.
        - If multiple matches are found, user must choose the correct ID.
        - Commits changes immediately after deletion.

    Example:
        delete_transactions(Title="Burger", Amount=250)

        delete_transactions(Type="Expense", Category="Food")
    """
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        extracted_rows = get_transactions_raw(Title , Amount , Type , Category , Mode)

        if len(extracted_rows) == 0:
            return "No matching rows found"

        if len(extracted_rows) == 1:
            transaction_id = extracted_rows[0][0] #extracts id from the whole row 

        else:
            matches = "\n".join(
                f"ID {r[0]}: '{r[2]}' | Rs.{r[3]} | {r[4]} | {r[5]}"
                for r in extracted_rows
            )
            return f"""
        I found multiple matching transactions.

        Please tell me which ID you want to delete.

        {matches}
        """

        cursor.execute(
            "DELETE FROM transactions WHERE id=?",
            (transaction_id,)
        )

        conn.commit()
        return "Deleted successfully"
    finally:
        if conn:
            conn.close()

In [ ]:
@tool

def update_transactions(
    Title: str,         # used to FIND the transaction
    field: str,           # column to update: Title, Amount, Category, Mode, Type
    new_value: str,       # new value to set
    Amount: int = None,   # optional extra filter to narrow search
    Category: str = None,
    Mode: str = None,
    Type: str = None,
) -> str:
    """
    Update an existing transaction in the database.

    Use this function to correct or modify transaction details such as
    title, amount, category, payment mode, or transaction type.

    Args:
        Title (str): Transaction title used to find matching records.
            Example: "Pizza", "Salary"

        field (str): Column name to update.
            Allowed values:
            - Title
            - Amount
            - Category
            - Mode
            - Type

        new_value (str): New value to assign to the selected field.

        Amount (int, optional): Extra filter to narrow matching records.

        Category (str, optional): Extra category filter.

        Mode (str, optional): Extra payment mode filter.

        Type (str, optional): Extra transaction type filter.
            Allowed values: "Income", "Expense"

    Returns:
        str: Status message after update.

    Behavior:
        - Validates allowed update fields.
        - If no record matches, returns an error message.
        - If one record matches, updates automatically.
        - If multiple records match, user selects the correct ID.
        - Commits changes immediately after update.

    Example:
        update_transactions(
            Title="Pizza",
            field="Amount",
            new_value="350"
        )

        update_transactions(
            Title="Salary",
            field="Category",
            new_value="Monthly Income"
        )
    """
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        allowed_fields = ["Title", "Amount", "Category", "Mode", "Type"]

        if field not in allowed_fields:
            return "Invalid Field"

        extracted_rows = get_transactions_raw(
        Title=Title,
        Amount=Amount,
        Type=Type ,
        Category=Category,
        Mode = Mode
        
    )

        if len(extracted_rows) == 0:
            return "No matching rows found"

        if len(extracted_rows) == 1:
            transaction_id = extracted_rows[0][0]

        else:
            matches = "\n".join(
                f"ID {r[0]}: '{r[2]}' | Rs.{r[3]} | {r[4]} | {r[5]}"
                for r in extracted_rows
            )
            return f"Multiple matches found. Ask the user which ID to update:\n{matches}"


        cursor.execute(
            f"UPDATE Transactions SET {field} = ? WHERE Id = ?",
            (new_value, transaction_id)
        )

        conn.commit()

        return "Transaction Updated Successfully"
    finally:
        if conn:
            conn.close()    

In [ ]:
@tool
def get_savings(
    Month: str = None,
    DateFrom: str = None,
    DateTo: str = None,
) -> str:
    """
    Calculate net savings (Income - Expenses).

    Use this tool when the user asks:
    - total savings
    - how much money was saved
    - savings for a month
    - savings for a year
    - savings between dates
    - net savings

    Args:
        Month (str, optional):
            Month in YYYY-MM format.

            Example:
            '2026-01'

        DateFrom (str, optional):
            Start date in YYYY-MM-DD format.

            Example:
            '2023-01-01'

        DateTo (str, optional):
            End date in YYYY-MM-DD format.

            Example:
            '2023-12-31'

    Returns:
        str:
            A formatted savings summary.

    Examples:
        get_savings()

        get_savings(
            Month='2026-01'
        )

        get_savings(
            DateFrom='2023-01-01',
            DateTo='2023-12-31'
        )
    """
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()

        where = "1=1"
        values = []

        if Month:
            where += " AND strftime('%Y-%m', Date) = ?"
            values.append(Month)

        if DateFrom:
            where += " AND Date >= ?"
            values.append(DateFrom)

        if DateTo:
            where += " AND Date <= ?"
            values.append(DateTo)

        query = f"""
        SELECT
            COUNT(*),

            COALESCE(SUM(
                CASE
                    WHEN Type='Income'
                    THEN CAST(Amount AS REAL)
                END
            ), 0)

            -

            COALESCE(SUM(
                CASE
                    WHEN Type='Expense'
                    THEN CAST(Amount AS REAL)
                END
            ), 0)

        FROM transactions
        WHERE {where}
        """

        cursor.execute(query, values)

        count, savings = cursor.fetchone()

        if count == 0:
            return "No transactions found."

        savings = round(savings, 2)

        if Month:
            return f"Your savings for {Month} are ₹{savings:,.2f}"

        if DateFrom and DateTo:
            return f"Your savings from {DateFrom} to {DateTo} are ₹{savings:,.2f}"

        return f"Your total savings are ₹{savings:,.2f}"
    finally:
        if conn:
            conn.close()

In [ ]:
@tool
def create_goal(
    Title: str,
    Target_amount: int,
    Deadline: str = None,           # FIX 5 — made optional; no deadline is valid
    Saved_amount: int = 0,
    Status: str = "Active",
) -> str:
    """
    Create a new financial savings goal.

    Use this function to add goals such as buying a laptop,
    travel planning, emergency fund, bike purchase, or any
    future savings target.

    Started_At is automatically set to today's date.
    No need to provide it manually.

    Args:
        Title (str): Short name of the goal.
            Example: "Buy Laptop", "Goa Trip", "Emergency Fund"

        Target_amount (int): Total money required to complete the goal.

        Deadline (str, optional): Target completion date in YYYY-MM-DD format.
            Example: "2026-12-31"
            Leave blank if no deadline is specified.

        Saved_amount (int, optional): Amount already saved.
            Defaults to 0.

        Status (str, optional): Current goal status.
            Allowed values: Active, Completed, Paused, Failed
            Defaults to 'Active'.

    Returns:
        str: Success message after creating the goal.

    Example:
        create_goal(Title="Buy Laptop", Target_amount=70000, Deadline="2026-12-31")
        create_goal(Title="Emergency Fund", Target_amount=50000)  # no deadline
    """
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        Started_at = date.today()

        cursor.execute(
            "INSERT INTO Goals (Title, Started_At, Deadline, Target_Amount, Saved_Amount, Status) VALUES (?,?,?,?,?,?)",
            (Title, Started_at, Deadline, Target_amount, Saved_amount, Status)
        )

        conn.commit()

        return "Goal created successfully"
    finally:
        if conn:
            conn.close()

In [ ]:
def get_goals_raw(
    Title: str = None,
    Started_at: str = None,
    Deadline: str = None,
    Target_amount: int = None,
    Saved_amount: int = None,
    Status: str = None,
) -> list:
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        query = "SELECT * FROM Goals WHERE 1=1"
        values = []

        if Title:
            query += " AND LOWER(Title) LIKE LOWER(?)"
            values.append(f"%{Title}%")

        if Started_at:
            query += " AND Started_At LIKE ?"
            values.append(f"%{Started_at}%")

        if Deadline:
            query += " AND Deadline LIKE ?"
            values.append(f"%{Deadline}%")

        if Target_amount is not None:
            query += " AND Target_Amount = ?"
            values.append(Target_amount)

        if Saved_amount is not None:
            query += " AND Saved_Amount = ?"
            values.append(Saved_amount)

        if Status:
            query += " AND Status LIKE ?"
            values.append(f"%{Status}%")

        cursor.execute(query, values)
        return cursor.fetchall()
    finally:
        if conn:
            conn.close()


In [ ]:
@tool
def get_goals(
    Title: str = None,
    Started_at: str = None,
    Deadline: str = None,
    Target_amount: int = None,
    Saved_amount: int = None,
    Status: str = None,
) -> list:
    """
    Retrieve financial goals using optional filters.

    Use this function to view all goals or search for specific
    goals based on title, dates, target amount, saved amount,
    or current status.

    All filters are optional. If no filters are provided,
    all saved goals are returned.

    Args:
        Title (str, optional): Search by goal title.
            Partial matches are allowed.
            Example: "Laptop", "Trip"

        Started_at (str, optional): Filter by goal start date
            in YYYY-MM-DD format.

        Deadline (str, optional): Filter by deadline date
            in YYYY-MM-DD format.

        Target_amount (int, optional): Filter by exact target amount.

        Saved_amount (int, optional): Filter by exact saved amount.

        Status (str, optional): Filter by goal status.
            Allowed values:
            - Active
            - Completed
            - Paused
            - Failed

    Returns:
        list: Matching goal records from the database.

    Behavior:
        - If filters are provided, only matching goals are returned.
        - If no filters are provided, all goals are returned.
        - Useful for progress tracking and goal management.

    Example:
        get_goals(Status="Active")

        get_goals(Title="Laptop")

        get_goals(Deadline="2026-12-31")
    """
    return get_goals_raw(
        Title,
        Started_at,
        Deadline,
        Target_amount,
        Saved_amount,
        Status
    )

In [ ]:
@tool
def delete_goal(
    Title: str = None,
    Started_at: str = None,
    Deadline: str = None,
    Target_amount: int = None,
    Saved_amount: int = None,
    Status: str = None,
) -> str:
    """
    Delete a financial goal using optional filters.

    Use this function to remove goals that are no longer needed,
    created by mistake, duplicated, or cancelled.

    The function first searches for matching goals and then deletes
    the selected record.

    Args:
        Title (str, optional): Search by goal title.
            Partial matches are allowed.
            Example: "Laptop", "Trip"

        Started_at (str, optional): Filter by start date
            in YYYY-MM-DD format.

        Deadline (str, optional): Filter by deadline date
            in YYYY-MM-DD format.

        Target_amount (int, optional): Filter by exact target amount.

        Saved_amount (int, optional): Filter by exact saved amount.

        Status (str, optional): Filter by goal status.
            Allowed values:
            - Active
            - Completed
            - Paused
            - Failed

    Returns:
        str: Status message after deletion.

    Behavior:
        - If no matching goal is found, returns an error message.
        - If exactly one match is found, deletes automatically.
        - If multiple matches are found, user selects the correct ID.
        - Commits changes immediately after deletion.

    Example:
        delete_goal(Title="Laptop")

        delete_goal(Status="Completed")

        delete_goal(Deadline="2026-12-31")
    """
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        extracted_goals = get_goals_raw(
            Title=Title,
            Started_at=Started_at,
            Deadline=Deadline,
            Target_amount=Target_amount,
            Saved_amount=Saved_amount,
            Status=Status,
        )

        if len(extracted_goals) == 0:
            return "No matching rows found"

        if len(extracted_goals) == 1:
            goal_id = extracted_goals[0][0]

        else:
            matches = "\n".join(
                f"ID {r[0]}: '{r[1]}' | Target Rs.{r[4]} | Saved Rs.{r[5]} | {r[6]}"
                for r in extracted_goals
            )
            return f"Multiple matches found. Ask the user which ID to delete:\n{matches}"


        cursor.execute("DELETE FROM Goals WHERE id=?", (goal_id,))

        conn.commit()

        return "Deleted successfully"
    finally:
        if conn:
            conn.close()

In [ ]:
@tool
def update_goals(
    Title: str,
    field: str,
    new_value: str,
    Started_at: str = None,
    Deadline: str = None,
    Target_amount: int = None,
    Saved_amount: int = None,
    Status: str = None,
) -> str:
    """
    Update an existing financial goal.

    Use this function to modify goal details such as title,
    deadline, target amount, saved amount, or current status.

    Title is used to find the matching goal record.
    Additional filters can be used to narrow the search.

    Args:
        Title (str): Goal title used to find matching records.
            Example: "Buy Laptop", "Goa Trip"

        field (str): Column name to update.
            Allowed values:
            - Title
            - Started_at
            - Deadline
            - Target_Amount
            - Saved_Amount
            - Status

        new_value (str): New value to assign to the selected field.

        Started_at (str, optional): Extra filter using start date
            in YYYY-MM-DD format.

        Deadline (str, optional): Extra filter using deadline date
            in YYYY-MM-DD format.

        Target_amount (int, optional): Extra filter using exact target amount.

        Saved_amount (int, optional): Extra filter using exact saved amount.

        Status (str, optional): Extra filter using goal status.
            Allowed values:
            - Active
            - Completed
            - Paused
            - Failed

    Returns:
        str: Status message after update.

    Behavior:
        - Validates allowed update fields.
        - If no matching goal is found, returns an error message.
        - If exactly one match is found, updates automatically.
        - If multiple matches are found, user selects the correct ID.
        - Commits changes immediately after update.

    Example:
        update_goals(
            Title="Buy Laptop",
            field="Saved_Amount",
            new_value="25000"
        )

        update_goals(
            Title="Goa Trip",
            field="Status",
            new_value="Completed"
        )
    """
    conn = None
    try :
        conn = get_connection()
        cursor = conn.cursor()
        allowed_fields = ["Title","Started_At","Deadline","Target_Amount","Saved_Amount","Status"]

        if field not in allowed_fields:
            return "Invalid Field"

        extracted_goals = get_goals_raw(
            Title=Title,
            Started_at=Started_at,
            Deadline=Deadline,
            Target_amount=Target_amount,
            Saved_amount=Saved_amount,
            Status=Status,
        )

        if len(extracted_goals) == 0:
            return "No matching rows found"

        if len(extracted_goals) == 1:
            goal_id = extracted_goals[0][0]

        else:
            matches = "\n".join(
                f"ID {r[0]}: '{r[1]}' | Target Rs.{r[4]} | Saved Rs.{r[5]} | {r[6]}"
                for r in extracted_goals
            )
            return f"Multiple matches found. Ask the user which ID to update:\n{matches}"


        # FIX 6 — Saved_Amount supports both increment (+500) and absolute (5000)
        if field == "Saved_Amount" and str(new_value).startswith("+"):
            cursor.execute(
                "UPDATE Goals SET Saved_Amount = Saved_Amount + ? WHERE Id = ?",
                (int(str(new_value).lstrip("+")), goal_id)
            )
        else:
            cursor.execute(
                f"UPDATE Goals SET {field} = ? WHERE Id = ?",
                (new_value, goal_id)
            )

        conn.commit()

        return "Goal Updated Successfully"
    finally:
        if conn:
            conn.close()

In [ ]:
SYSTEM_PROMPT = SYSTEM_PROMPT = """
You are MoneyWise AI. Use tools only. Never answer from memory. Never fake data.

RULES:
- No Amount → ask before adding. Never guess.
- No data returned → "No records found."
- Unclear request → ask one question only.
- Use ₹. Dates: YYYY-MM-DD. Month: YYYY-MM.

DATE RULES:

- today → current date
- yesterday → current date - 1
- this month → current YYYY-MM
- last month → previous YYYY-MM

Convert all dates to:
YYYY-MM-DD

INTENT MAP:
show/list/fetch/get/history    → get_transactions / get_goals / get_savings
spent/bought/paid/got/received → add_transaction
update/change/edit/rename      → update_transactions / update_goals
delete/remove/erase            → delete_transactions / delete_goal

EXAMPLES:
show all transactions                        → get_transactions()
show all expenses                            → get_transactions(Type="Expense")
show food expenses                           → get_transactions(Type="Expense", Category="Food")
show January transactions                    → get_transactions(Month="2026-01")
show expenses above 1000                     → get_transactions(Type="Expense", AmountMin=1000)
show transactions 1 Jan to 31 Mar           → get_transactions(DateFrom="2026-01-01", DateTo="2026-03-31")
show my savings                              → get_savings()
how much did I save in January               → get_savings(Month="2026-01")
bought pizza for 300                         → add_transaction(Title="Pizza Purchase", Amount=300, Type="Expense", Category="Food", Mode="Online")
spent 699 on a trimmer                       → add_transaction(Title="Trimmer Purchase", Amount=699, Type="Expense", Category="Shopping", Mode="Online")
paid electricity bill 1200                   → add_transaction(Title="Electricity Bill", Amount=1200, Type="Expense", Category="Bills", Mode="Online")
received salary 25000                        → add_transaction(Title="Salary", Amount=25000, Type="Income", Category="Salary", Mode="Bank Transfer")
father gave me 500                           → add_transaction(Title="Pocket Money", Amount=500, Type="Income", Category="Pocket Money", Mode="Cash")
invested 5000 in SIP                         → add_transaction(Title="SIP Investment", Amount=5000, Type="Expense", Category="Investment", Mode="Online")
update pizza amount to 400                   → update_transactions(Title="Pizza Purchase", field="Amount", new_value="400")
change Netflix mode to UPI                   → update_transactions(Title="Netflix", field="Mode", new_value="UPI")
delete pizza transaction                     → delete_transactions(Title="Pizza Purchase")
show all goals                               → get_goals()
show active goals                            → get_goals(Status="Active")
save for iPhone 15 worth 80000 by Dec 2026  → create_goal(Title="Buy iPhone 15", Target_amount=80000, Deadline="2026-12-31", Saved_amount=0, Status="Active")
add 5000 to iPhone goal                      → update_goals(Title="iPhone 15", field="Saved_Amount", new_value="+5000")
mark Goa trip as completed                   → update_goals(Title="Goa Trip", field="Status", new_value="Completed")
delete Goa trip goal                         → delete_goal(Title="Goa Trip")

CATEGORIES:
Expense: Food, Groceries, Transport, Education, Shopping, Entertainment, Healthcare, Bills, Travel, Subscription, Investment, Other
Income:  Salary, Pocket Money, Freelancing, Business, Gift, Refund, Cashback, Investment, Other

pizza/burger/lunch          → Food       petrol/uber/auto       → Transport
vegetables/milk/fruits      → Groceries  movie/gaming           → Entertainment
books/stationery            → Education  medicine/doctor        → Healthcare
clothes/shoes/electronics   → Shopping   electricity/internet   → Bills
flight/hotel                → Travel     netflix/spotify        → Subscription
stocks/SIP/mutual fund      → Investment salary                 → Salary
father/mother gave money    → Pocket Money  freelance payment   → Freelancing
refund/returned money       → Refund     rent                   → Bills

Mode default (add only): Online. Omit mode for fetch/update/delete.
Type: Expense = money out. Income = money in.

DELETE/UPDATE SAFETY RULES:

- Never update or delete multiple rows automatically.
- If multiple matches exist:
    show matching IDs
    ask which ID to use.

If a tool response asks for clarification, ID selection, or additional user input:
- stop immediately
- wait for the next user message
- never call another tool automatically

TOOLS:
add_transaction(Title, Amount, Type, Category, Mode)
get_transactions(Title, Amount, Type, Category, Mode, Date, Month, DateFrom, DateTo, AmountMin, AmountMax)
update_transactions(Title, field, new_value, Amount, Category, Mode, Type)
delete_transactions(Title, Amount, Type, Category, Mode)
get_savings(Month, DateFrom, DateTo)
create_goal(Title, Target_amount, Deadline, Saved_amount, Status)
get_goals(Title, Started_at, Deadline, Target_amount, Saved_amount, Status)
update_goals(Title, field, new_value, Started_at, Deadline, Target_amount, Saved_amount, Status)
delete_goal(Title, Started_at, Deadline, Target_amount, Saved_amount, Status)

RESPONSES:
add_transaction success      → "Transaction added successfully."
delete_transactions success  → "Transaction deleted successfully."
update_transactions success  → "Transaction updated successfully."
create_goal success          → "Goal created successfully."
delete_goal success          → "Goal deleted successfully."
update_goals success         → "Goal updated successfully."
Multiple matches found       → show list with IDs, ask "Which ID should I use

GOAL RULES:

- "save for", "target", "goal", "by <date>"
    usually indicate create_goal.

- "added money to goal"
    indicates update_goals.

- Regular spending/income statements
    indicate add_transaction.

Use generate_and_execute_sql only for analytical queries
Analytical queries include:
- totals
- comparisons
- trends
- highest/lowest
- averages
- category breakdowns
- monthly summaries
- percentages
- statistics

Do NOT use CRUD tools for analytical questions.


SQL TOOL RULES:

- Use generate_and_execute_sql only for analytical, reporting, aggregation, trend, comparison, summary, or calculation queries.
- Never use it for add/update/delete operations.
- Examples:
    - "Which category did I spend most on?"
    - "Show monthly expenses trend"
    - "What is my highest expense?"
    - "How much did I spend on food this year?"
    - "Compare income vs expenses"
    - "Top 5 biggest transactions"
    - "Average monthly spending"

- After tool response:
    - If result is empty → "No records found."
    - Summarize results naturally for the user.
    - Keep responses concise.
    - Never expose raw SQL unless user asks.
"""

In [ ]:
agent = create_agent(
    model=client,
    tools=[ generate_and_execute_sql ,  add_transaction , delete_transactions , update_transactions , get_transactions , get_savings, create_goal , delete_goal , update_goals , get_goals ],
    system_prompt= SYSTEM_PROMPT
)

In [ ]:

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "delete pens purchase transaction"
            }
        ]
    },
    config={"recursion_limit": 5}
)

GraphRecursionError: Recursion limit of 5 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [ ]:
print(response["messages"][-1].content)

Transactions

In [ ]:
cursor.execute("SELECT * FROM Transactions")
rows = cursor.fetchall()

df = pd.DataFrame(rows, columns=[
    "id",
    "date",
    "title",
    "amount",
    "type",
    "category",
    "payment_method"
])

display(df)
# convert the data base into dataframe to make visuzlation and also get insights 


In [ ]:
df.iloc[18]

Goals

In [ ]:
cursor.execute("SELECT * FROM Goals")
rows = cursor.fetchall()

df2 = pd.DataFrame(rows, columns=[
    "id",
    "Title",
    "Started_at",
    "Deadline",
    "Target_Amount",
    "Saved_Amount",
    "Status"
])

display(df2)

In [ ]:
cursor.execute("SELECT COUNT(*) FROM transactions")
print(cursor.fetchone())